In [ ]:
#training_v2_run_a3

In [2]:
"""
model_diagnostics.py
================================
Comprehensive diagnostic visualization for TSFNet_With_fNIRS_2D
(architecture used in training_v2_run_a3.ipynb -- the best-result run).

Architecture reference (from the training script):
  - EEG branch:    BiLSTM (hidden=64, bidirectional) -> last timestep -> eeg_feat (64-dim)
  - fNIRS branch:  nirs_fc (Linear, GAP-based) -> nirs_feat (64-dim)
  - Fusion branch: CAFE cross-attention -> final_feat (64-dim), final_nirs (64-dim)
  - Classifiers:   fc_eeg / fc_fnirs / fc_fusion =
                       Sequential(Linear(64,32), ELU, Dropout, Linear(32,3))
                   each preceded by self.dropout(feature) OUTSIDE the Sequential
                   (i.e. dropout is applied twice: once before, once inside)
  - forward() without return_attention returns an 8-tuple:
        (final_out, eeg_out, fnirs_out, fusion_out,
         eeg_feat_map, Phi2_bc_loss, final_nirs, final_feat)

Analyses performed:
  1.  Confusion matrix (normalized + raw) per branch
  2.  Per-class accuracy per branch
  3.  PCA of 64-dim features (train vs test)
  4.  t-SNE of 64-dim features (train vs test) + misclassified overlay
  5.  Train vs Test feature distribution shift (mean +/- std per dimension)
  6.  Per-class centroid distance matrix (class separability)
  7.  Logit confidence & entropy histograms
  8.  Silhouette score & Davies-Bouldin index per feature space
  9.  Input gradient saliency (EEG) -- with cudnn RNN backward fix
  10. Text summary report

HOW TO USE
----------
Call this at the end of run_training(), right before train_dataset.cleanup():

    run_diagnostics(
        model        = model,
        train_loader = train_loader,
        test_loader  = test_loader,
        device       = device,
        session_id   = test_session,
        output_dir   = os.path.join(PROCESSED_DATA_PATH, "diagnostics")
    )
"""

import os
import gc
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (confusion_matrix, silhouette_score,
                              davies_bouldin_score, classification_report)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

CLASS_NAMES = ["0-back", "2-back", "3-back"]
COLORS      = ["#2a78d6", "#e24b4a", "#1baf7a"]


# ──────────────────────────────────────────────────────────────────────────
# 1. Feature extraction
#    Hooks on fc_eeg[0] / fc_fnirs[0] / fc_fusion[0]: since fc_eeg =
#    Sequential(Linear(64,32), ELU, Dropout, Linear(32,3)), inp[0] of the
#    hook on fc_eeg[0] is the 64-dim vector fed into the first Linear --
#    i.e. self.dropout(eeg_feat) (dropout is applied before the Sequential
#    in this architecture, see forward()). This is the correct feature
#    point for PCA/t-SNE.
# ──────────────────────────────────────────────────────────────────────────

def _extract_features(model, loader, device):
    """
    One forward pass over `loader`, collecting:
      feat_eeg, feat_fnirs, feat_fusion : each (N, 64)
      feat_combined                     : (N, 192)
      logits                            : (N, 3) -- final weighted logits
      labels                            : (N,)
    eeg_inputs is intentionally NOT stored (RAM) -- saliency reads directly
    from test_data["_loader"] instead.
    """
    model.eval()
    feat_store = {"eeg": [], "fnirs": [], "fusion": []}

    def _hook_factory(key):
        def _hook(module, inp, out):
            feat_store[key].append(inp[0].detach().cpu().numpy())
        return _hook

    h1 = model.fc_eeg[0].register_forward_hook(_hook_factory("eeg"))
    h2 = model.fc_fnirs[0].register_forward_hook(_hook_factory("fnirs"))
    h3 = model.fc_fusion[0].register_forward_hook(_hook_factory("fusion"))

    all_logits, all_labels = [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in loader:
            out = model(eeg_b.to(device), nirs_b.to(device))
            # out[0] = final_out (8-tuple, return_attention=False)
            all_logits.append(out[0].cpu().numpy())
            all_labels.append(lbl_b.numpy())

    h1.remove(); h2.remove(); h3.remove()

    feat_eeg    = np.concatenate(feat_store["eeg"],    axis=0)
    feat_fnirs  = np.concatenate(feat_store["fnirs"],  axis=0)
    feat_fusion = np.concatenate(feat_store["fusion"], axis=0)
    logits      = np.concatenate(all_logits,           axis=0)
    labels      = np.concatenate(all_labels,           axis=0)
    feat_combined = np.concatenate([feat_eeg, feat_fnirs, feat_fusion], axis=1)

    del feat_store, all_logits, all_labels
    gc.collect()

    return {
        "feat_eeg": feat_eeg, "feat_fnirs": feat_fnirs, "feat_fusion": feat_fusion,
        "feat_combined": feat_combined, "logits": logits, "labels": labels,
        "eeg_inputs": None,   # deliberately None -- saliency reads from _loader
    }


# ──────────────────────────────────────────────────────────────────────────
# 2. Per-branch logits (separate pass -- hooks on the LAST Linear of each head)
# ──────────────────────────────────────────────────────────────────────────

def _collect_branch_logits(model, loader, device):
    branch_logits = {"eeg": [], "fnirs": [], "fusion": [], "final": []}

    def _hook_logit(key):
        def _h(m, inp, out):
            branch_logits[key].append(out.detach().cpu())
        return _h

    # [-1] = Linear(32,3), the final logit layer of each classifier head
    h1 = model.fc_eeg[-1].register_forward_hook(_hook_logit("eeg"))
    h2 = model.fc_fnirs[-1].register_forward_hook(_hook_logit("fnirs"))
    h3 = model.fc_fusion[-1].register_forward_hook(_hook_logit("fusion"))

    model.eval()
    with torch.no_grad():
        for eeg_b, nirs_b, _ in loader:
            out = model(eeg_b.to(device), nirs_b.to(device))
            branch_logits["final"].append(out[0].cpu())

    h1.remove(); h2.remove(); h3.remove()
    return {k: torch.cat(v, dim=0).numpy() for k, v in branch_logits.items()}


# ──────────────────────────────────────────────────────────────────────────
# 3. Confusion matrix
# ──────────────────────────────────────────────────────────────────────────

def _plot_confusion(branch_logits, labels, session_id, output_dir):
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    key_map = {"Final": "final", "EEG": "eeg", "fNIRS": "fnirs", "Fusion": "fusion"}

    for ax, name in zip(axes, ["Final", "EEG", "fNIRS", "Fusion"]):
        preds = np.argmax(branch_logits[key_map[name]], axis=1)
        cm    = confusion_matrix(labels, preds)
        cmn   = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        im    = ax.imshow(cmn, vmin=0, vmax=100, cmap="Blues")
        ax.set_xticks(range(3)); ax.set_yticks(range(3))
        ax.set_xticklabels(CLASS_NAMES, fontsize=9)
        ax.set_yticklabels(CLASS_NAMES, fontsize=9)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"{name}  (acc={cm.diagonal().sum()/cm.sum()*100:.1f}%)", fontsize=10)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{cmn[i,j]:.1f}%\n({cm[i,j]})",
                        ha="center", va="center",
                        color="white" if cmn[i,j] > 55 else "black", fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle(f"Confusion Matrices -- Session {session_id}", fontsize=13, y=1.01)
    plt.tight_layout()
    path = os.path.join(output_dir, f"S{session_id}_01_confusion.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 4. Per-class accuracy per branch
# ──────────────────────────────────────────────────────────────────────────

def _plot_per_class_acc(branch_logits, labels, session_id, output_dir):
    fig, ax = plt.subplots(figsize=(10, 5))
    x, width = np.arange(3), 0.2
    branch_colors = ["#2a78d6", "#e24b4a", "#1baf7a", "#f5a623"]
    key_map = {"Final": "final", "EEG": "eeg", "fNIRS": "fnirs", "Fusion": "fusion"}

    for bi, (bname, bcolor) in enumerate(zip(["Final", "EEG", "fNIRS", "Fusion"], branch_colors)):
        preds = np.argmax(branch_logits[key_map[bname]], axis=1)
        accs  = [(preds[labels == c] == c).mean() * 100 for c in range(3)]
        bars  = ax.bar(x + bi*width, accs, width, label=bname, color=bcolor,
                       alpha=0.85, edgecolor="white")
        for bar, v in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                    f"{v:.1f}", ha="center", fontsize=7.5)

    ax.set_xticks(x + 1.5*width); ax.set_xticklabels(CLASS_NAMES)
    ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 105)
    ax.axhline(33.3, ls="--", color="gray", lw=0.8, label="Chance")
    ax.legend(fontsize=9)
    ax.set_title(f"Per-class Accuracy per Branch -- Session {session_id}")
    plt.tight_layout()
    path = os.path.join(output_dir, f"S{session_id}_02_per_class_acc.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 5. PCA
# ──────────────────────────────────────────────────────────────────────────

def _plot_pca(train_data, test_data, session_id, output_dir):
    for feat_key, title in [("feat_eeg",      "EEG branch (64-dim, BiLSTM read-out)"),
                              ("feat_fnirs",    "fNIRS branch (64-dim, nirs_fc)"),
                              ("feat_combined", "All branches (192-dim)")]:
        X_tr, y_tr = train_data[feat_key], train_data["labels"]
        X_te, y_te = test_data[feat_key],  test_data["labels"]

        scaler = StandardScaler().fit(X_tr)
        pca    = PCA(n_components=2, random_state=42).fit(scaler.transform(X_tr))
        Z_tr   = pca.transform(scaler.transform(X_tr))
        Z_te   = pca.transform(scaler.transform(X_te))

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for ax, (Z, y, split) in zip(axes, [(Z_tr, y_tr, "Train"), (Z_te, y_te, "Test")]):
            for c, (cname, col) in enumerate(zip(CLASS_NAMES, COLORS)):
                mask = y == c
                ax.scatter(Z[mask,0], Z[mask,1], s=8, alpha=0.5, color=col, label=cname)
            ax.set_title(f"PCA -- {split} | {title}")
            ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
            ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
            ax.legend(markerscale=3, fontsize=8)

        plt.suptitle(f"Session {session_id} -- PCA Feature Space", fontsize=12)
        plt.tight_layout()
        tag  = feat_key.replace("feat_", "")
        path = os.path.join(output_dir, f"S{session_id}_03_pca_{tag}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 6. t-SNE + misclassified overlay
# ──────────────────────────────────────────────────────────────────────────

def _plot_tsne(test_data, session_id, output_dir, perplexity=30):
    for feat_key, title in [("feat_eeg",      "EEG branch (BiLSTM)"),
                              ("feat_fnirs",    "fNIRS branch (nirs_fc)"),
                              ("feat_combined", "All branches")]:
        X    = test_data[feat_key]
        y    = test_data["labels"]
        pred = np.argmax(test_data["logits"], axis=1)

        X_s = StandardScaler().fit_transform(X)
        n_max = min(3000, len(X_s))
        idx   = np.random.choice(len(X_s), n_max, replace=False)
        X_s, y, pred = X_s[idx], y[idx], pred[idx]

        print(f"    t-SNE {feat_key} ({n_max} samples)...", end="", flush=True)
        Z = TSNE(n_components=2, perplexity=perplexity, random_state=42,
                 max_iter=1000).fit_transform(X_s)
        print(" done")

        wrong = pred != y
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        ax = axes[0]
        for c, (cname, col) in enumerate(zip(CLASS_NAMES, COLORS)):
            mask = y == c
            ax.scatter(Z[mask,0], Z[mask,1], s=7, alpha=0.45, color=col, label=cname)
        ax.set_title("Colour = true class"); ax.legend(markerscale=3, fontsize=9)

        ax = axes[1]
        ax.scatter(Z[~wrong,0], Z[~wrong,1], s=6, alpha=0.35, color="#1baf7a", label="Correct")
        ax.scatter(Z[wrong,0], Z[wrong,1], s=10, alpha=0.7, color="#e24b4a",
                   label="Misclassified", marker="x")
        ax.set_title(f"Correct vs Misclassified (error {wrong.mean()*100:.1f}%)")
        ax.legend(markerscale=3, fontsize=9)

        fig.suptitle(f"t-SNE -- Session {session_id} -- {title}", fontsize=12)
        plt.tight_layout()
        tag  = feat_key.replace("feat_", "")
        path = os.path.join(output_dir, f"S{session_id}_04_tsne_{tag}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 7. Train vs Test distribution shift
# ──────────────────────────────────────────────────────────────────────────

def _plot_distribution_shift(train_data, test_data, session_id, output_dir):
    for feat_key, title in [("feat_eeg",   "EEG branch (64-dim, BiLSTM)"),
                              ("feat_fnirs", "fNIRS branch (64-dim, nirs_fc)")]:
        X_tr = train_data[feat_key]; X_te = test_data[feat_key]
        mu_tr, mu_te   = X_tr.mean(0), X_te.mean(0)
        std_tr, std_te = X_tr.std(0),  X_te.std(0)
        dims = np.arange(X_tr.shape[1])

        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        ax = axes[0]
        ax.plot(dims, mu_tr, color="#2a78d6", lw=1.2, label="Train mean")
        ax.plot(dims, mu_te, color="#e24b4a", lw=1.2, label="Test mean")
        ax.fill_between(dims, mu_tr-std_tr, mu_tr+std_tr, color="#2a78d6", alpha=0.15)
        ax.fill_between(dims, mu_te-std_te, mu_te+std_te, color="#e24b4a", alpha=0.15)
        ax.set_xlabel("Feature dimension"); ax.set_ylabel("Activation")
        ax.set_title(f"Mean +/- Std -- {title}"); ax.legend()

        ax = axes[1]
        shift = np.abs(mu_tr - mu_te)
        ax.bar(dims, shift, color="#ef9f27", width=0.8)
        ax.set_xlabel("Feature dimension")
        ax.set_ylabel("|Train mean - Test mean|")
        ax.set_title("Distribution shift per dimension (high = domain shift)")
        top5 = np.argsort(shift)[-5:]
        for d in top5:
            ax.axvline(d, color="red", lw=0.8, ls="--")
        ax.text(0.98, 0.95, f"Top-5 shifted dims: {sorted(top5.tolist())}",
                transform=ax.transAxes, ha="right", fontsize=8, color="red")

        plt.suptitle(f"Session {session_id} -- Distribution Shift", fontsize=12)
        plt.tight_layout()
        tag  = feat_key.replace("feat_", "")
        path = os.path.join(output_dir, f"S{session_id}_05_dist_shift_{tag}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 8. Class centroid distance matrix
# ──────────────────────────────────────────────────────────────────────────

def _plot_centroid_distance(train_data, test_data, session_id, output_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, (data, split) in zip(axes, [(train_data, "Train"), (test_data, "Test")]):
        X = data["feat_combined"]; y = data["labels"]
        centroids = np.array([X[y==c].mean(0) for c in range(3)])
        dist = np.array([[np.linalg.norm(centroids[i]-centroids[j]) for j in range(3)]
                         for i in range(3)])
        im = ax.imshow(dist, cmap="YlOrRd")
        ax.set_xticks(range(3)); ax.set_yticks(range(3))
        ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{dist[i,j]:.2f}", ha="center", va="center",
                        fontsize=10, fontweight="bold")
        plt.colorbar(im, ax=ax)
        ax.set_title(f"{split} -- Centroid Distance (larger = more separable)")

    plt.suptitle(f"Session {session_id} -- Class Centroid Distance (192-dim)", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, f"S{session_id}_06_centroid_dist.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 9. Confidence & Entropy
# ──────────────────────────────────────────────────────────────────────────

def _plot_confidence(test_data, session_id, output_dir):
    logits = test_data["logits"]; labels = test_data["labels"]
    probs   = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds   = np.argmax(logits, axis=1)
    correct = preds == labels

    max_prob = probs.max(axis=1)
    entropy  = -(probs * np.log(probs + 1e-9)).sum(axis=1)
    max_ent  = np.log(3)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(max_prob[correct],  bins=40, alpha=0.7, color="#1baf7a",
            label=f"Correct ({correct.sum()})", density=True)
    ax.hist(max_prob[~correct], bins=40, alpha=0.7, color="#e24b4a",
            label=f"Wrong ({(~correct).sum()})", density=True)
    ax.set_xlabel("Max softmax probability"); ax.set_ylabel("Density")
    ax.set_title("Confidence Distribution"); ax.legend()
    ax.axvline(0.5, color="gray", ls="--", lw=0.8)

    ax = axes[1]
    ax.hist(entropy[correct]/max_ent,  bins=40, alpha=0.7, color="#1baf7a",
            label="Correct", density=True)
    ax.hist(entropy[~correct]/max_ent, bins=40, alpha=0.7, color="#e24b4a",
            label="Wrong", density=True)
    ax.set_xlabel("Normalised entropy (0=certain, 1=uniform)")
    ax.set_ylabel("Density")
    ax.set_title("Prediction Uncertainty (Entropy)"); ax.legend()

    plt.suptitle(f"Session {session_id} -- Confidence & Uncertainty", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, f"S{session_id}_07_confidence.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 10. Cluster metrics
# ──────────────────────────────────────────────────────────────────────────

def _compute_cluster_metrics(data, session_id):
    results = {}
    for feat_key in ["feat_eeg", "feat_fnirs", "feat_combined"]:
        X = data[feat_key]; y = data["labels"]
        n   = min(2000, len(X))
        idx = np.random.choice(len(X), n, replace=False)
        Xs, ys = StandardScaler().fit_transform(X[idx]), y[idx]
        results[feat_key] = {
            "silhouette":     silhouette_score(Xs, ys, metric="euclidean"),
            "davies_bouldin": davies_bouldin_score(Xs, ys),
        }
    return results


# ──────────────────────────────────────────────────────────────────────────
# 11. Input Gradient Saliency (EEG)
#
#     cudnn RNN backward requires training mode. model.eval() blocks
#     LSTM's backward pass with:
#       "cudnn RNN backward can only be called in training mode"
#     Fix: switch to model.train() but freeze every BatchNorm layer (keep
#     it using its stored running statistics) so activations stay
#     equivalent to eval mode -- standard pattern for gradient-based
#     saliency on models containing RNNs.
# ──────────────────────────────────────────────────────────────────────────

def _plot_saliency(model, test_data, device, session_id, output_dir, n_samples=200):
    if "_loader" not in test_data or test_data["_loader"] is None:
        print("  [saliency] skipped -- no loader reference stored.")
        return

    model.train()
    for m in model.modules():
        if isinstance(m, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
            m.eval()   # BN stays in eval -- uses stored running_mean/var

    _grads = {c: [] for c in range(3)}
    collected = 0

    for eeg_b, nirs_b, lbl_b in test_data["_loader"]:
        if collected >= n_samples:
            break
        eeg_b  = eeg_b.to(device).float()
        nirs_b = nirs_b.to(device).float()
        lbl_np = lbl_b.numpy()
        eeg_b.requires_grad_(True)

        out = model(eeg_b, nirs_b)[0]   # final_out: (B, 3)

        for c in range(3):
            mask = (lbl_np == c)
            if mask.sum() == 0:
                continue
            if eeg_b.grad is not None:
                eeg_b.grad.zero_()
            model.zero_grad()
            score = out[mask, c].sum()
            score.backward(retain_graph=(c < 2))   # free graph on the last class
            if eeg_b.grad is not None:
                grad = eeg_b.grad.detach().cpu().abs()
                _grads[c].append(grad[mask].mean(0))

        eeg_b = eeg_b.detach()
        collected += len(lbl_np)

    model.eval()   # restore eval mode after saliency

    if not any(_grads[c] for c in range(3)):
        print("  [saliency] no gradients collected -- skipped.")
        return

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    for c, ax in enumerate(axes):
        if not _grads[c]:
            ax.set_ylabel(f"|Gradient|\n{CLASS_NAMES[c]}", fontsize=9)
            continue
        g      = torch.stack(_grads[c]).mean(0)
        g_time = g.reshape(-1, g.shape[-1]).mean(0).numpy()
        t      = np.linspace(0, 10, len(g_time))   # 10s EEG window
        ax.plot(t, g_time, color=COLORS[c], lw=1.0)
        ax.fill_between(t, 0, g_time, color=COLORS[c], alpha=0.3)
        ax.set_ylabel(f"|Gradient|\n{CLASS_NAMES[c]}", fontsize=9)
        ax.set_xlim(0, 10)

    axes[-1].set_xlabel("Time within window (seconds)")
    plt.suptitle(f"Session {session_id} -- Input Gradient Saliency (EEG)\n"
                 "(which time points drive predictions)", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, f"S{session_id}_08_saliency.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ──────────────────────────────────────────────────────────────────────────
# 12. Text report
# ──────────────────────────────────────────────────────────────────────────

def _write_report(test_data, cluster_metrics, session_id, output_dir):
    logits = test_data["logits"]; labels = test_data["labels"]
    preds  = np.argmax(logits, axis=1)

    lines = [
        f"DIAGNOSTIC REPORT -- Session {session_id}", "=" * 60,
        "\n[Classification Report -- Final Branch]",
        classification_report(labels, preds, target_names=CLASS_NAMES),
        "\n[Cluster Separability Metrics]",
        f"{'Feature':20s}  {'Silhouette':>12}  {'Davies-Bouldin':>15}",
        "-" * 52,
    ]
    for k, v in cluster_metrics.items():
        tag = k.replace("feat_", "")
        lines.append(f"{tag:20s}  {v['silhouette']:12.4f}  {v['davies_bouldin']:15.4f}")
    lines += [
        "\nNote: Silhouette in [-1,1] -- higher is better separable.",
        "      Davies-Bouldin >= 0    -- lower is better separable.",
        "\n[Model Architecture Notes]",
        "  EEG branch:   BiLSTM (hidden=64, bidirectional) -> last timestep -> Linear(128->64)",
        "  fNIRS branch: nirs_fc (Linear, GAP-based) -> 64-dim",
        "  Fusion:       CAFE cross-attention -> 64-dim",
        "  Classifier:   Linear(64->32) -> ELU -> Dropout -> Linear(32->3)",
        "  Loss:         CrossEntropy + EFGF regularization + CAFE (PCC) regularization",
    ]

    path = os.path.join(output_dir, f"S{session_id}_report.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"  [saved] {path}")
    print("\n".join(lines))


# ──────────────────────────────────────────────────────────────────────────
# MAIN ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────

def run_diagnostics(model, train_loader, test_loader, device, session_id, output_dir):
    """
    Parameters
    ----------
    model        : trained TSFNet_With_fNIRS_2D instance
    train_loader : DataLoader for the training sessions
    test_loader  : DataLoader for the test session
    device       : torch.device
    session_id   : int (1, 2, or 3)
    output_dir   : str -- folder where all PNGs and the report are saved
    """
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  DIAGNOSTICS -- Session {session_id}")
    print(f"{'='*60}")

    print("\n[1/9] Extracting features -- train set...")
    train_data = _extract_features(model, train_loader, device)

    print("[2/9] Extracting features -- test set...")
    test_data  = _extract_features(model, test_loader, device)
    test_data["_loader"] = test_loader   # for saliency

    print("[3/9] Collecting per-branch logits...")
    branch_logits = _collect_branch_logits(model, test_loader, device)

    labels = test_data["labels"]

    print("[4/9] Confusion matrices...")
    _plot_confusion(branch_logits, labels, session_id, output_dir)

    print("[5/9] Per-class accuracy...")
    _plot_per_class_acc(branch_logits, labels, session_id, output_dir)

    print("[6/9] PCA...")
    _plot_pca(train_data, test_data, session_id, output_dir)

    print("[7/9] t-SNE (1-2 min)...")
    _plot_tsne(test_data, session_id, output_dir)

    print("[8/9] Distribution shift...")
    _plot_distribution_shift(train_data, test_data, session_id, output_dir)

    _plot_centroid_distance(train_data, test_data, session_id, output_dir)
    _plot_confidence(test_data, session_id, output_dir)

    print("[+] Cluster metrics...")
    cluster_metrics = _compute_cluster_metrics(test_data, session_id)

    print("[+] Gradient saliency (EEG -- with cudnn RNN fix)...")
    _plot_saliency(model, test_data, device, session_id, output_dir)

    print("[+] Writing report...")
    _write_report(test_data, cluster_metrics, session_id, output_dir)

    del train_data, test_data, branch_logits, labels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\n  All diagnostics saved -> {output_dir}\n")


In [1]:
#Tsfnet cross session
#version4
# ============================================================================
# RUN A3 (isolated experiment): EEG-branch temporal read-out via BiLSTM.
#
# BASE: training_v2_run_a1_10s.py (10s EEG windows, 200 Hz -> (16,16,2000)).
#       Reads MULTIMODAL_PREPROCESSED_RUN_A1_10s, identical to Run A1.
#
# THE PROBLEM A3 FIXES
# --------------------
# In the baseline model the EEG classifier feature was produced as:
#       eeg_feat_map = ex2.mean(dim=2)                 # temporal GAP
#       eeg_feat     = <Linear 512->64>(eeg_feat_map.flatten(1))
# Because the temporal Global Average Pooling happens BEFORE the linear layer,
# every time-resolved pattern in the EEG (for instance the P300 deflection at
# 300-500 ms, or the theta build-up that separates 2-back from 3-back) is
# averaged out before the classifier can use it. The EEG branch therefore only
# ever saw a static 32x4x4 spatial map.
#
# THE CHANGE (EEG BRANCH ONLY)
# ----------------------------
#   __init__ : the old EEG Linear read-out (512->64) is REMOVED and replaced by
#              self.eeg_lstm      = nn.LSTM(512, 64, bidirectional=True)
#              self.eeg_lstm_proj = nn.Linear(128, 64)
#   forward(): eeg_feat_map = ex2.mean(dim=2) is KEPT UNCHANGED, because it is
#              still consumed in two other places:
#                1. L_EFGF: pearson_correlation_loss(eeg_feat_map, Phi2_bc_loss)
#                2. the shape-alignment test against Phi2_bc_loss
#              Only eeg_feat now comes from the BiLSTM read-out of ex2 instead
#              of from the Linear read-out applied to eeg_feat_map.
#
# Everything else is byte-for-byte the baseline: the fNIRS branch, the fusion
# branch, EFGF1/EFGF2, CAFE, the loss function (L_class + L_efgf + L_cafe),
# the two-stage training schedule, the trial-level 4:1 split, the optimizer,
# and the cooling-break/cleanup logic -- per single-variable isolation.
#
# Output files carry the A3 suffix so they never collide with A1/B3 results:
#   best_tsfnet_A3_session{n}.pth, results_A3_session{n}.json,
#   cross_session_A3_summary.json
#
# NOTE ON ENCODING: this file is pure ASCII. All non-ASCII comments inherited
# from the baseline were translated to English; no executable line was changed
# by that translation.
# ============================================================================
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
import time  # NEW (cooling-break + memmap-cleanup patch): used for the inter-session cooling delay
import glob  # NEW (cooling-break + memmap-cleanup patch): used to find leftover t_mmap_*.npy files
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import uuid  # NEW (Windows/Jupyter mmap-lock fix): used to generate a unique per-run mmap filename, see MMapHybridDataset.__init__
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
# NOTE (path consistency): this MUST point at the same OUTPUT_PATH folder that the fixed
# Preprocessing script writes to (VPxxx subfolders containing *_eeg_windows.npy, *_fnirs_windows.npy,
# *_labels.npy, *_session_ids.npy, *_trial_ids.npy, *_metadata.json). If you re-run Preprocessing,
# update this path to match its OUTPUT_PATH, or copy/point this at the folder that actually has
# {subject}_trial_ids.npy in it -- that file is now produced by the fixed Preprocessing script and
# is REQUIRED below by the trial-level 4:1 split.
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\A1b_10s_epoch-level-threshold0.3"  # CHANGED (Run A1-revised): reads the 10s-EEG-window data, does not touch the 3s-window baseline folder

ALL_SUBJECTS = list(range(1, 27))

# CROSS-SESSION CONFIGURATION (per TSFNet paper Sec. 3.6):
# Instead of splitting subjects into train/test (cross-subject), for each subject
# one session is used as test and the two remaining sessions as train.
# Hold-Out: 3 runs -- each run uses a different session as test.
# Final result = mean test accuracy over the three runs.
# This follows Sec. 3.6 of the paper, which evaluates the cross-session setting.
# Source: VPxxx_session_ids.npy produced by generate_session_ids.py, which stores
# for every window the value 1, 2 or 3 (session number).

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(RANDOM_SEED)

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=all sessions, [1,2]=sessions 1 and 2 only
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: which sessions should be loaded
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    # CROSS-SESSION: filter based on session_ids.npy
                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found -- using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,  # CROSS-SESSION: None or a boolean mask
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]

        # FIX (Windows/Jupyter mmap-lock bug): the previous version always used a FIXED filename
        # per dataset_name (e.g. t_mmap_train_set_e.npy), and tried to os.remove() it here before
        # re-creating it. On Windows, np.memmap keeps an OS-level file handle open for as long as
        # the numpy array object (and anything referencing it, e.g. a leftover variable in a
        # Jupyter notebook, an old `train_dataset` in `Out[]`, or a not-yet-garbage-collected
        # Dataset from a previous cell run) is alive. If that old memmap hasn't been released yet
        # when this __init__ runs again (very common in Jupyter, where re-running a cell before
        # the previous dataset object is deleted/garbage-collected is the normal workflow),
        # os.remove() raises PermissionError, which the old code caught with a bare
        # `except: pass` and SILENTLY ignored -- then proceeded to open the SAME still-locked file
        # in mode='w+' and stream NEW data into it. Depending on OS/driver behavior this can
        # either raise a confusing low-level error, or -- worse -- silently succeed while the old
        # memmap object still holds a view onto the same bytes, producing stale/mixed data with
        # no warning at all.
        #
        # FIX: give every dataset instance construction its own **unique** filename (process id +
        # random uuid), so there is never a name collision with any previous memmap file (locked
        # or not) and the fragile "remove-then-recreate" step is no longer needed at all -- this
        # removes the entire class of Windows file-lock problems rather than trying to catch it.
        run_tag = f"{os.getpid()}_{uuid.uuid4().hex[:8]}"  # NEW: unique per dataset construction, avoids any Windows file-lock collision
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_{run_tag}_e.npy")   # CHANGED: unique filename (was fixed name)
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_{run_tag}_n.npy")   # CHANGED: unique filename (was fixed name)

        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            # CHANGED: no os.remove() needed here anymore -- the unique run_tag guarantees this
            # exact path was never used before, so nothing to delete or silently overwrite. If,
            # for some astronomically unlikely reason, the path already exists (e.g. leftover from
            # a previous crashed run that reused a pid), fail LOUDLY instead of silently reusing
            # a foreign file, since silently writing into an unexpected pre-existing file is
            # exactly the kind of silent data-corruption risk this fix is meant to eliminate.
            if os.path.exists(f_path):  # CHANGED: was "if exists: try remove, except: pass"
                raise FileExistsError(  # NEW: fail loudly instead of silently reusing/overwriting
                    f"[MMAP COLLISION] Unexpected pre-existing mmap file (should be impossible with "
                    f"a unique run_tag): '{f_path}'. Please delete this file manually and re-run, "
                    f"or check for a stale/leftover file from an earlier crashed run."
                )
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        #self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 2000))  # CHANGED: (N,28,600) -> (N,16,16,600)
        
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 21, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        n_files = len(self.valid_files)  # NEW (progress-print patch): needed for the "[k/n_files]" progress line below
        for file_idx, f_info in enumerate(self.valid_files):  # CHANGED: was "for f_info in self.valid_files:" -- enumerate() added for the progress line
            s, e = f_info['start_idx'], f_info['end_idx']
            # CROSS-SESSION: if session_mask exists, load only the corresponding windows
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            # NEW (progress-print patch): emits output after every subject so the browser/Jupyter
            # front-end keeps seeing activity during this long, otherwise-silent streaming loop,
            # instead of appearing to hang and the kernel connection being dropped as dead.
            print(f"  [{file_idx+1:02d}/{n_files}] {f_info['subject']} "  # NEW
                  f"({e-s} windows, cumulative {e}/{self.total_windows})", flush=True)  # NEW
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        # FIX (Windows/Jupyter mmap-lock bug, continued): flush explicitly BEFORE deleting the
        # array references, so any buffered writes are committed to disk before we try to release
        # the OS file handle. On Windows the handle is only released once ALL references to the
        # underlying np.memmap object are gone AND garbage-collected -- del + gc.collect() below
        # is what actually drops the handle; flush() alone does not close the file.
        try:  # NEW: flush before del, defensive (matches __init__'s already-flushed state, but
              # NEW: cheap and safe to repeat here in case cleanup() is ever called before the
              # NEW: final flush at the end of __init__, e.g. after a partial/failed load)
            self.X_eeg.flush()
            self.X_nirs.flush()
        except Exception:
            pass
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try:
                    os.remove(f)
                except PermissionError as e:
                    # CHANGED (Windows/Jupyter mmap-lock bug): previously a bare `except: pass`
                    # that silently swallowed this error. Because every dataset instance now gets
                    # a UNIQUE filename (see __init__), a failure here can NEVER cause another
                    # dataset to silently read/write stale or mixed data -- the orphaned file is
                    # simply never referenced again. Still, silently hiding a real OS error is bad
                    # practice, so this now prints an explicit, actionable warning instead of
                    # disappearing into a bare except/pass.
                    print(f"[WARNING] Could not delete temporary mmap file (Windows file lock still "
                          f"held, likely by a lingering Python reference in this Jupyter session): "
                          f"'{f}'\n"
                          f"  This file is now orphaned and safe to ignore (it will not be reused "
                          f"by any other dataset, since every run gets a unique filename), but you "
                          f"may want to delete it manually later to free disk space, or restart the "
                          f"Jupyter kernel to release the lock.")
                except Exception as e:
                    # NEW: any other unexpected OS-level deletion error is also surfaced instead of
                    # being silently swallowed, for the same reasoning as above.
                    print(f"[WARNING] Could not delete temporary mmap file '{f}': {e}")

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    # Upgraded version: full Transformer block with FFN
    # TSFNet paper: CAFE alone has ~2.77M parameters (Table 6: 3.33M total - 0.57M without CAFE = 2.77M).
    # The previous version was only MultiheadAttention(64,nhead=4), far lighter than the paper.
    # This version:
    #   1. raises d_model to 256 (from 64) -- the main driver of the parameter increase
    #   2. adds an FFN with dim_feedforward=512 -- like a standard Transformer
    #   3. nhead=8 instead of 4 -- the paper emphasizes "multi-head attention"
    # Parameters per CrossAttention block with d_model=256, ffn=512:
    #   MHA: 256*256*3 + 256*256 ~= 262K
    #   FFN: 256*512 + 512*256 ~= 262K
    #   LayerNorm x 2: ~1K
    #   Total per block: ~525K -- two blocks (cafe_attn + cafe_attn2) ~= 1.05M
    #   With projection layers (nirs_seq_proj, gate layers): ~= 1.3-1.5M for the whole CAFE
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # FFN -- like a standard Transformer encoder block
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        # Cross-attention with residual
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        # FFN with residual
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        # ================= RUN A3 CHANGE (EEG temporal read-out) =================
        # REMOVED (Run A3): the EEG Linear read-out nn.Linear(32 * 4 * 4, 64)
        # That layer consumed eeg_feat_map, i.e. the tensor AFTER temporal GAP,
        # so every temporal pattern (e.g. the P300 deflection at 300-500 ms) was
        # averaged away before the classifier ever saw it. A3 replaces that read-out
        # with a BiLSTM that runs over the temporal axis of ex2, while eeg_feat_map
        # itself is left completely untouched for L_EFGF and the EFGF alignment.
        self.eeg_lstm = nn.LSTM(
            input_size=32 * 4 * 4,  # = 512  (C*H*W of ex2)
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0.0
        )
        self.eeg_lstm_proj = nn.Linear(128, 64)  # 128 = 64*2 because bidirectional
        # =========================================================================

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # -- Fusion Branch (paper Table 1, third column) -----------------------
        # The paper has an independent Conv3d branch that takes the EEG tensor but whose
        # weights are learned completely separately from the EEG branch. Its purpose is to
        # prepare the EEG representation for fusion with fNIRS, not pure EEG classification.
        # kernel/stride parameters are exactly like the EEG branch (Table 1):
        #   Conv1: in (1,600,16,16) -> out (16,100,8,8)
        #   Conv2: in (16,100,8,8)  -> out (32,50,4,4)
        # Output after GAP over the temporal axis: F_fusion in R^{32x4x4}
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)
        # ----------------------------------------------------------------------

        # -- Two-stage EFGF -- corrected per paper Eq.2 ------------------------
        # Paper: the attention map is built from the full feature VOLUME (temporal dim kept):
        #   Phi = alpha*sigma(GAP(Conv3d(F_fnirs))) + (1-alpha)*sigma(GAP(Conv3d(F_eeg)))
        # Conv3d is applied to the full volume, then GAP removes the temporal dimension.
        # Result: a spatial attention map (h,w) that has absorbed temporal information.
        #
        # EFGF1 -- on the feature volume after Conv1: (batch,16,100,8,8)
        # Table 1: kernel (2,2,6), stride (1,1,1) -> output (batch,1,8,8,~95), then temporal GAP
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))
        #
        # EFGF2 -- on the feature volume after Conv2: (batch,32,50,4,4)
        # Table 1: kernel (2,2,3), stride (1,1,1) -> output (batch,1,4,4,~48), then temporal GAP
        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))
        #
        # final projection of F_fusion (spatial 32x4x4) to 64-d for the classifier
        # CAFE operates on the spatial map, so flatten is only needed for fc_fusion
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)
        # ----------------------------------------------------------------------

        # CAFE
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        # CAFE -- d_model=128
        # CHANGED: reduced from 256 to 128.
        # With 26 subjects in a cross-subject scenario, a 1.5M-parameter CAFE overfit badly
        # (train acc reached 99% while val acc stayed at 54%).
        # 128-d is a good middle ground: more than the initial 64-d (more capacity),
        # less than 256-d (less overfit). CAFE at d_model=128: ~350K vs ~1.39M for 256-d.
        CAFE_DIM = 128
        #self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=21)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers -- two dense layers (paper Section 3.4: "two dense layers followed by softmax")
        # Glorot et al. (2011) JMLR: a hidden layer between feature (64-d) and output (3 classes)
        # performs better than a direct mapping.
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        # FIX (paper Eq.4 -- Paper-Compliant Fix 1/3: Weighted combination):
        # Paper Eq.4: "w1, w2, w3 are trainable weight coefficients obtained via a sigmoid
        # function applied to initialized parameters".
        # Previously: F.softmax(stack(w1,w2,w3)), which forces the weights to sum to 1 and adds
        # a constraint that does not exist in the paper.
        # Now: torch.sigmoid(wi) for each weight independently, exactly per Eq.4.
        # sigmoid is in (0,1) so each weight lies between 0 and 1, but their sum may differ from 1.
        # Paper: "Mean denotes the averaging operation" -> Mean(w1*y_eeg, w2*y_fusion, w3*y_fnirs)
        # i.e. a plain average (not a normalized weighted sum) of the three weighted predictions.
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs, return_attention=False):  # CHANGED (XAI patch): added optional return_attention param, per patch_instructions.md
        batch_size = eeg.size(0)

        # -- EEG branch --------------------------------------------------------
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)
        # temporal GAP -- kept EXACTLY as before (needed by L_EFGF and EFGF alignment)
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) -- kept for L_EFGF
        # ================= RUN A3 CHANGE (EEG temporal read-out) =================
        # OLD: eeg_feat = <Linear 512->64>(eeg_feat_map.flatten(1))  # (batch,64), no temporal info
        # NEW: eeg_feat comes from a BiLSTM over the temporal axis of ex2, so temporal
        # structure survives into the EEG classifier. eeg_feat_map is NOT modified.
        B, C, T, H, W = ex2.shape                            # T ~= 166 for the 10s window
        eeg_seq = ex2.permute(0, 2, 1, 3, 4).reshape(B, T, C * H * W)  # (batch,166,512)
        lstm_out, _ = self.eeg_lstm(eeg_seq)                 # (batch,166,128)
        eeg_feat = self.eeg_lstm_proj(lstm_out[:, -1, :])    # (batch,64) -- last timestep
        # =========================================================================

        # -- Fusion branch -- Conv1 (before EFGF1) -----------------------------
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)
        # NOTE: fx2 is not computed yet -- it must be computed after EFGF1 (fx1_updated)
        # Paper Figure 4: F_hat_fusion (the EFGF1 output) enters Conv2 of the fusion branch,

        # -- fNIRS branch ------------------------------------------------------
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # (batch*11,16,15,8,8)
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))     # (batch*11,32,8,4,4)
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        # GAP over segment and time for the fNIRS feature map
        nirs_feat_map = ny2.mean(dim=(1, 3))                 # (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))   # (batch,64)
        y_spatial = ny2.mean(dim=(3, 4, 5))                  # (batch,11,32) -- for CAFE

        # -- EFGF1: on the full feature VOLUME after Conv1 ---------------------
        # Paper Eq.2: Phi = alpha*sigma(GAP(Conv3d(F_fnirs))) + (1-alpha)*sigma(GAP(Conv3d(F_eeg)))
        # fNIRS: fold/unfold so it is applied independently on each segment
        ny1_vol = ny1.mean(dim=1)                            # (batch,16,15,8,8) -- mean over segments
        # Conv3d on the full volume (temporal dim kept) -> temporal GAP -> spatial attention map
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)            # (batch,1,T',H',W') -- T' shrunk
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)        # (batch,1,T'',H',W')
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))     # (batch,1,H',W') -- temporal GAP
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))    # (batch,1,H',W')
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1            # (batch,1,H',W')
        # Eq.1: Fhat_fus1 = gamma1*F^eeg_1 + (1-gamma1)*F_fus1 + Phi1 (x) F_fus1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)                          # (batch,1,1,H',W')
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]
        # (batch,16,T,H',W') -- F_hat_fusion, the EFGF1 output

        # -- Fusion branch -- Conv2 (FIX Paper-Compliant Fix 2/3: EFGF chain) --
        # Previously: fx2 = fus_conv2(fx1)  <- EFGF1 had no effect on the fusion branch
        # Now: fx2 = fus_conv2(fx1_updated)  <- per paper Figure 4
        # Paper Figure 4: F_hat_fusion (the EFGF1 output) enters Conv2 of the fusion branch.
        # This guarantees EFGF1 really participates in the fusion-branch processing chain
        # and that gradient flows from Conv2 back to EFGF1 -- previously that link was cut.
        # FIX: fx1_updated may have smaller spatial dims than fx1 (because of padding
        # truncation inside EFGF1). adaptive_avg_pool3d is used to align them so that
        # fus_conv2 works with in_channels=16 and the correct kernel.
        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )  # (batch,16,100,8,8) -- same shape as the original fx1
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))  # FIX: fx1 -> fx1_updated_aligned
        fx2 = self.fus_drop2(fx2)
        # (batch,32,50,4,4) -- the full volume is preserved for EFGF2

        # -- EFGF2: on the full feature VOLUME after Conv2 ---------------------
        ny2_vol = ny2.mean(dim=1)                            # (batch,32,8,4,4)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)            # (batch,1,T',H',W')
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)        # (batch,1,T'',H',W')
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))    # (batch,1,H',W')
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2            # (batch,1,H',W') -- attention map
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)                          # (batch,1,1,H',W')
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]      # (batch,32,T,H',W')
        # temporal GAP after EFGF2 -> spatial feature map for CAFE and the classifier
        fus_feat_map = fx2_updated.mean(dim=2)               # (batch,32,H',W')
        # H',W' may have become smaller than 4x4 after the EFGF Conv3d
        # adaptive pool so it matches fusion_proj, which expects 32x4x4
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))  # (batch,32,4,4)
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))  # (batch,64)

        # For L_EFGF (Eq.12): PCC between eeg_feat_map and Phi2 (size-matched)
        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)          # (batch,32,H',W')
        # if the spatial dims differ from eeg_feat_map, align them
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # -- CAFE Block -- in the 128-d space ----------------------------------
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)            # (batch,11,128)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)           # (batch,11,128)
        fused_256     = self.fusion_cafe_proj(fused_feat)        # (batch,128)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)           # (batch,128)
        # Cross-Attn 2: fusion <- fNIRS
        fused_seq_256  = fused_256.unsqueeze(1)                  # (batch,1,128)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        # Cross-Attn 1: fNIRS <- fusion
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        # back to 64-d for the classifier
        final_feat = self.cafe_out_proj(final_feat_256)          # (batch,64)
        final_nirs = self.cafe_out_proj2(final_nirs_256)         # (batch,64)

        # -- Classifier --------------------------------------------------------
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        # FIX (paper Eq.4 -- Paper-Compliant Fix 1/3: Weighted combination):
        # Previously: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        #         final_out = w[0]*eeg_out + w[1]*fnirs_out + w[2]*fusion_out
        # softmax forces the weights to sum to 1 -- that is not in the paper.
        # Paper Eq.4: "w1, w2, w3 obtained via a sigmoid function"
        #             "Mean denotes the averaging operation"
        # i.e. each weight passes through sigmoid independently, then a weighted mean is taken.
        # Now: sigmoid per weight, Mean for the combination -- exactly Eq.4.
        w1 = torch.sigmoid(self.w1)  # sigmoid per Eq.4
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        # FIX (3-class adaptation): removed /3 -- dividing by 3 rescaled the logits and
        # shrank the gradient by 3x (gradient of w*logit -> gradient/3), which made
        # learning slow and unstable. It had no effect on argmax but artificially
        # kept the loss value high (test: loss without /3 = 0.817, with /3 = 0.933).
        # A direct weighted sum is also mathematically cleaner: CrossEntropyLoss operates
        # on logits and needs no prior normalization.
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out  # FIX: /3 removed

        # CHANGED (XAI patch, per patch_instructions.md): when return_attention=True, also return
        # the raw EFGF attention maps Phi1 (after EFGF1, shape (batch,1,H',W')) and Phi2 (after
        # EFGF2, shape (batch,1,H',W')) as two extra tuple items -- these are what
        # xai_analysis.py's extract_attention_maps() reads via outputs[-2], outputs[-1]. Note this
        # is Phi2 itself, NOT Phi2_bc_loss (which is Phi2 already expanded/pooled to match
        # eeg_feat_map's shape for the PCC regularization loss -- that item stays at index 5,
        # unchanged, so run_training()'s existing unpacking of the 8-tuple is unaffected).
        if return_attention:  # NEW
            return (final_out, eeg_out, fnirs_out, fusion_out,
                    eeg_feat_map, Phi2_bc_loss,
                    final_nirs, final_feat,
                    Phi1, Phi2)   # NEW: only appended when return_attention=True is explicitly requested
        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,   # Fe^fnirs -- for L_CAFE: nirs_feat_map in the training loop
                final_feat)   # Fe^fusion -- for L_CAFE: shape (batch,64) like final_nirs


# NEW (overfitting-fix experiment -- stronger L2 weight decay, per chat history): the previous
# run showed the classic Krogh & Hertz (1992) overfitting signature -- best val accuracy reached
# at epoch 1-2, then train accuracy climbs to 85-90% while val stagnates/declines, indicating the
# ~180k-parameter model is memorizing subject-specific noise rather than learning generalizable
# cognitive-load features, given the unfavorable parameter-to-independent-sample ratio (~259:1)
# in this subject-independent 3-class n-back setup.
#
# Krogh, A., & Hertz, J. A. (1992). "A simple weight decay can improve generalization."
# Advances in Neural Information Processing Systems (NeurIPS), 4, 950-957.
# -- shows analytically that L2 weight decay directly shrinks the effective weight-space volume
#    the optimizer can reach, reducing variance (overfitting) WITHOUT changing the network's
#    representational capacity (unlike pruning or reducing channel counts, which was already
#    tried in this project and found to be mostly noise-level -- see project report Section 4).
#
# This project's code already uses weight_decay=2e-2 (a 40x increase over the original 5e-4),
# confirming this direction was already identified as helpful; the natural next step -- exactly
# as recommended in the prior chat analysis -- is NOT to introduce a new regularization concept,
# but to test whether an EVEN STRONGER L2 penalty further improves generalization, following
# Krogh & Hertz's finding that there is a task-dependent optimal weight-decay coefficient (too
# small under-regularizes, too large under-fits) that must be found empirically, i.e. via a small
# grid search rather than a single guess.
#
# WEIGHT_DECAY_GRID lists the current baseline (2e-2) alongside two stronger candidates (5e-2,
# 1e-1) suggested in the prior analysis. Everything else (seed, data split, architecture,
# optimizer type, learning rate, loss weights, scheduler, early-stopping patience) is held
# EXACTLY fixed across the three runs below, so any difference in outcome can be attributed
# specifically to the weight_decay coefficient -- following the same single-variable-isolation
# discipline used throughout this project (e.g. Run C's bundled-filter mistake, which the project
# report explicitly flags as a methodological lesson).
#WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]  # paper: Adam without weight decay (default = 0)

LAMBDA_EEG = 0.2     # Eq. 6 lambda_eeg -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_FNIRS = 0.2   # Eq. 6 lambda_fnirs -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_EFGF_REG = 1.0   # Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
LAMBDA_CAFE_REG = 1.0   # Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

# TWO-STAGE TRAINING (paper Sec. 3.6):
# Stage 1: train on 80% of train set, early-stop on 20% val, patience=50, max=300 epochs.
#          Best checkpoint (highest val acc) is restored at end of stage 1.
# Stage 2: retrain on FULL train set (no val split), stop when train loss drops below
#          the best stage-1 train loss achieved, max=200 epochs.
STAGE1_MAX_EPOCHS = 300   # paper Sec. 3.6 says max 300 for stage 1
STAGE1_PATIENCE   = 50    # paper says patience 50
STAGE2_MAX_EPOCHS = 200   # paper Sec. 3.6 says max 200 for stage 2

# FIX (paper Sec. 3.6 -- Paper-Compliant Fix 3/3: Window-level 4:1 val split):
# Previously: subject-level split -- 2 whole subjects were held out for val.
# Paper Sec. 3.6: "the training set was randomly split into a first-stage training subset
# and a validation set in a 4:1 ratio."
# This split is over time windows, not over subjects.
# Now: the full TRAIN_SUBS set is used for stage 1, but 20% of the windows are held
# out at random as val -- exactly per the paper's method.
# Note: S1_VAL_SUBS and S1_TRAIN_SUBS are no longer used; the split happens inside
# run_training with SubsetRandomSampler over indices.
# This makes val acc come from windows of the same train subjects (as in the paper).


# NEW: the full train+evaluate pipeline (previously the top-level "MAIN PIPELINE" block) is now
# wrapped in a function parameterized by weight_decay, so it can be run once per grid-search
# candidate with everything else held fixed. The body is otherwise UNCHANGED from the original
# script -- no architecture, loss, optimizer type, learning rate, scheduler, or data-loading logic
# was touched, per the instruction to implement only the weight-decay experiment.
def run_training(weight_decay, device, test_session=3):  # CROSS-SESSION: test_session = 1, 2 or 3
    train_sessions = [s for s in [1, 2, 3] if s != test_session]  # CROSS-SESSION: the two remaining sessions
    print("\n" + "#" * 70)
    print(f"# CROSS-SESSION: train on sessions {train_sessions}  |  test on session {test_session}")
    print("#" * 70)

    from torch.utils.data import SubsetRandomSampler

    # CROSS-SESSION: all 26 subjects, but only windows belonging to train_sessions or test_session
    train_dataset = MMapHybridDataset(ALL_SUBJECTS, "Train_Set",
                                       session_filter=train_sessions,
                                       min_windows_threshold=200)   # CROSS-SESSION
    test_dataset  = MMapHybridDataset(ALL_SUBJECTS, "Test_Set",
                                       session_filter=[test_session],
                                       min_windows_threshold=100)   # CROSS-SESSION

    # FIX: window-level 4:1 split over TRAIN_SUBS -- per paper Sec. 3.6
    #n_total_train = len(train_dataset)
    #indices = list(range(n_total_train))
    #random.shuffle(indices)  # seed=42 already set -- reproducible
    #split = int(n_total_train * 0.8)  # 80% train, 20% val -- the paper's 4:1 ratio
    #s1_train_indices = indices[:split]
    #s1_val_indices   = indices[split:]
    #print(f"[Cross-Session] Window-level 4:1 split: "
     #     f"{len(s1_train_indices)} train windows, {len(s1_val_indices)} val windows "
      #    f"(from {len(ALL_SUBJECTS)} subjects, sessions {train_sessions}, ratio={len(s1_train_indices)/len(s1_val_indices):.1f}:1)")
    # -- Trial-level 4:1 split (replaces the window-level split) ---------------
    # Logic: shuffle the trials, give 80% to train and 20% to val
    # No window from a given trial ends up in both train and val
    # -> leakage caused by overlap of adjacent windows is eliminated
    
    # collect trial_ids from all subjects (train sessions only)
    # NOTE (trial_ids dependency): this part requires the file {subject}_trial_ids.npy, which the
    # corrected version of Preprocessing (extract_synchronized_windows -> process_subject) now
    # actually creates and saves. If your preprocessed data was produced by the old Preprocessing
    # version (without trial_ids), you must re-run preprocessing to generate this file, otherwise
    # the np.load line below will stop with FileNotFoundError.
    all_trial_ids_global = []  # global trial id for each window in train_dataset
    
    window_offset = 0
    for f_info in train_dataset.valid_files:
        subject = f_info['subject']
        sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
        trial_ids_path = os.path.join(sub_folder, f"{subject}_trial_ids.npy")
        
        trial_ids_full = np.load(trial_ids_path)  # all 27 trials
        
        # only windows belonging to the train sessions
        if f_info['session_mask'] is not None:
            trial_ids_filtered = trial_ids_full[f_info['session_mask']]
        else:
            trial_ids_filtered = trial_ids_full
        
        n_windows_this_subject = f_info['end_idx'] - f_info['start_idx']
        assert len(trial_ids_filtered) == n_windows_this_subject, \
            f"{subject}: trial_ids mismatch ({len(trial_ids_filtered)} vs {n_windows_this_subject})"
        
        # global trial id = (subject_index * 100) + local_trial_id
        # multiplying by 100 guarantees that trials of different subjects never get mixed
        sub_idx = ALL_SUBJECTS.index(int(subject[2:]))
        global_trial_ids = sub_idx * 100 + trial_ids_filtered
        all_trial_ids_global.extend(global_trial_ids.tolist())
        window_offset += n_windows_this_subject
    
    all_trial_ids_global = np.array(all_trial_ids_global)
    assert len(all_trial_ids_global) == len(train_dataset), \
        f"trial_ids length mismatch: {len(all_trial_ids_global)} vs {len(train_dataset)}"
    
    # shuffle at trial level (not window level)
    unique_trials = np.unique(all_trial_ids_global)
    rng = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(unique_trials)
    
    split_trial = int(len(unique_trials) * 0.8)
    train_trials = set(unique_trials[:split_trial])
    val_trials   = set(unique_trials[split_trial:])
    
    s1_train_indices = [i for i, t in enumerate(all_trial_ids_global) if t in train_trials]
    s1_val_indices   = [i for i, t in enumerate(all_trial_ids_global) if t in val_trials]
    
    print(f"[Trial-level split] {len(unique_trials)} trials total | "
          f"{len(train_trials)} train trials ({len(s1_train_indices)} windows) | "
          f"{len(val_trials)} val trials ({len(s1_val_indices)} windows) | "
          f"ratio={len(s1_train_indices)/max(len(s1_val_indices),1):.1f}:1")
    print(f"[Trial-level split] Zero overlap guaranteed: "
          f"no window from a val trial appears in train")




    s1_train_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_train_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler for 80% of the windows
    s1_val_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_val_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler for 20% of the windows
    train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # CROSS-SESSION optimizer: paper Sec. 3.6 uses Adam with lr=0.001 (no weight_decay)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    #scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    # FIX (3-class adaptation): class-weighted CrossEntropyLoss for the n-back imbalance
    # The paper worked on binary (balanced) tasks -- label_smoothing alone was enough there.
    # In the 3-class n-back setting, 0-back usually has more windows (easier, longer),
    # which without correction biases the model toward the majority class and makes it
    # learn 2-back and 3-back less well -- exactly the problem reported for cross-subject
    # BCI classification (Lotte et al., 2018).
    # Solution: inverse-frequency weighting -- lower-count classes receive larger weights.
    # Weights are computed from train_dataset.y (not test) to avoid data leakage.
    # label_smoothing=0.1 is still kept -- the two work together:
    #   class_weight: corrects the model's bias toward the majority class
    #   label_smoothing: reduces overconfidence
    # Source: King & Zeng (2001) -- inverse frequency weighting for imbalanced classification
    _train_labels   = train_dataset.y                                        # numpy array over all train windows
    _class_counts   = np.bincount(_train_labels, minlength=3).astype(float) # [n_0back, n_2back, n_3back]
    _class_weights  = 1.0 / (_class_counts + 1e-6)                          # inverse frequency
    _class_weights  = _class_weights / _class_weights.sum() * 3             # normalize: mean weight = 1
    _weight_tensor  = torch.tensor(_class_weights, dtype=torch.float32).to(device)
    print(f"[3-class] Class counts (train): 0-back={int(_class_counts[0])}, "
          f"2-back={int(_class_counts[1])}, 3-back={int(_class_counts[2])}")
    print(f"[3-class] Class weights:        0-back={_class_weights[0]:.3f}, "
          f"2-back={_class_weights[1]:.3f}, 3-back={_class_weights[2]:.3f}")
    # label_smoothing=0.1: Mueller et al. (2019) NeurIPS "When Does Label Smoothing Help?"
    # In cross-subject settings train/test distributions differ -- label smoothing improves
    # calibration and prevents overconfident predictions, which lead directly to overfitting.
    #criterion = nn.CrossEntropyLoss(weight=_weight_tensor, label_smoothing=0.1)  # FIX: class weights added
    criterion = nn.CrossEntropyLoss(weight=_weight_tensor)

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_A3_session{test_session}.pth')

    # ------------------------------------------------------------------ #
    # STAGE 1: train on 80% of TRAIN windows, early-stop on 20% val      #
    # per paper Sec. 3.6: window-level 4:1 split from the train sessions #
    # ------------------------------------------------------------------ #
    print(f"STAGE 1 -- training on {len(s1_train_indices)} windows (80% of train sessions {train_sessions}), "
          f"val on {len(s1_val_indices)} windows (20%) | "
          f"max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE} | window-level 4:1 split (paper)")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_val_train_loss = float('inf')  # train loss at the best-val epoch
    stage1_final_train_loss    = float('inf')  # train loss at the last epoch -- fallback when best val is epoch 1
    train_acc = 0.0
    history = {'epoch': [], 'train_acc': [], 'val_acc': [], 'train_loss': []}  # NEW (XAI patch, per patch_instructions.md): records Stage-1 learning curve data for xai_analysis.py's plot_learning_curves()

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_indices)
        avg_train_loss = train_loss / len(s1_train_loader)
        stage1_final_train_loss = avg_train_loss  # updated at every epoch

        # stage-1 validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_indices)
        #scheduler.step(val_acc)

        # NEW (XAI patch, per patch_instructions.md): record this epoch's stats. This does not
        # change any training behavior -- it only saves values that were already being computed
        # and printed, into a dict for later plotting by xai_analysis.py's plot_learning_curves().
        history['epoch'].append(epoch)          # NEW
        history['train_acc'].append(train_acc*100)  # NEW
        history['val_acc'].append(val_acc*100)       # NEW
        history['train_loss'].append(avg_train_loss)  # NEW

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stage1_best_val_train_loss = avg_train_loss
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    # Restore best stage-1 weights before stage 2
    model.load_state_dict(torch.load(ckpt_path))
    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)
    #stage2_target = stage1_best_val_train_loss
    print(f"\n[S1] Best checkpoint restored.")
    print(f"[S1] best-val-epoch loss: {stage1_best_val_train_loss:.4f} | final-epoch loss: {stage1_final_train_loss:.4f}")
    print(f"[S1] Stage-2 target loss: {stage2_target:.4f} (min of the two -- ensures stage 2 actually trains)")

    print(f"\nSTAGE 2 -- retraining on full {len(train_dataset)} samples ({len(ALL_SUBJECTS)} subjects, sessions {train_sessions}) "
          f"| max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage2_target:.4f}")

    # Fresh optimizer for stage 2 -- same Adam lr=0.001 as in the paper
    optimizer2 = optim.Adam(model.parameters(), lr=0.001)
    #scheduler2  = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', factor=0.5, patience=5)

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)
        #scheduler2.step(avg_s2_loss)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage2_target:.4f}")

        if avg_s2_loss < stage2_target:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< target {stage2_target:.4f}")
            torch.save(model.state_dict(), ckpt_path)
            break
    else:
        # Reached max epochs without meeting loss criterion -- save anyway
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    # FINAL MODEL TEST & EVALUATION
    print(f"\n RUNNING FINAL TEST EVALUATION (test_session={test_session})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (Mean(w1*eeg, w2*fusion, w3*fnirs))")

    # FIX: learned weights now come from sigmoid (not softmax)
    with torch.no_grad():
        learned_w1 = torch.sigmoid(model.w1).item()
        learned_w2 = torch.sigmoid(model.w2).item()
        learned_w3 = torch.sigmoid(model.w3).item()
    print(f"\nLearned final-combination weights (after sigmoid, paper Eq.4): "
          f"w_eeg={learned_w1:.3f}, w_fusion={learned_w2:.3f}, w_fnirs={learned_w3:.3f}")

    #overfit_gap = (train_acc * 100) - test_acc
    #print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")
    # FIX: overfit_gap used to be computed from the final Stage-2 train_acc, which is always ~100%
    # (Stage 2 has no validation supervision and, by the paper's design, fits train as far as possible).
    # That made overfit_gap always look large (~45-50pp) regardless of whether the leakage in the
    # val split had been fixed -- i.e. the metric never showed real "overfitting", only the
    # designed difference between Stage 1 (with val) and Stage 2 (without val).
    # Better metric: compare best_val_acc (Stage 1, leakage-free trial-level val) with test_acc.
    # If those two are close the model really generalizes; a large gap means val (drawn from the
    # train sessions) is not a good proxy for a completely unseen session (test).
    overfit_gap_train_vs_test = (train_acc * 100) - test_acc  # still stored for reference / historical comparison
    overfit_gap_val_vs_test = (best_val_acc * 100) - test_acc  # NEW: the primary, more meaningful metric
    
    print(f"\n[Reference, not primary] Stage-2 final train_acc - test_acc: {overfit_gap_train_vs_test:+.2f}pp "
          f"(NOTE: Stage 2 has no validation signal, so this is expected to be large "
          f"by design and does NOT indicate real overfitting)")
    print(f"Generalization gap (best Stage-1 val_acc - test_acc): {overfit_gap_val_vs_test:+.2f} percentage points "
          f"(this is the meaningful overfitting/generalization indicator, "
          f"since val is now trial-level split with no leakage)")


    results = {
        'test_session': test_session,
        'train_sessions': train_sessions,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': learned_w1,
            'w_fusion': learned_w2,
            'w_fnirs': learned_w3,
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap_train_vs_test': overfit_gap_train_vs_test,  # NEW: historical reference, misleading on its own
        'overfitting_gap_val_vs_test': overfit_gap_val_vs_test,        # NEW: primary and meaningful metric
    }
    results['history'] = history  # NEW (XAI patch, per patch_instructions.md): Stage-1 per-epoch train/val accuracy + train loss, read by xai_analysis.py's plot_learning_curves()
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_A3_session{test_session}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    # NEW (visualization): run full model diagnostics before the train/test
    # datasets and model are released. Wrapped in try/except so that a
    # diagnostics failure never causes loss of the (already-saved) training
    # results.
    try:
        run_diagnostics(
            model        = model,
            train_loader = train_loader,
            test_loader  = test_loader,
            device       = device,
            session_id   = test_session,
            output_dir   = os.path.join(PROCESSED_DATA_PATH, "diagnostics")
        )
    except Exception as diag_err:
        print(f"[WARNING] run_diagnostics failed: {diag_err}")
        print("  Training results are saved. Diagnostics skipped.")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# MAIN PIPELINE -- Cross-Session Hold-Out (per TSFNet paper Sec. 3.6)
# Runs three times: each time one session as test, two sessions as train.
# Final result = mean test accuracy over the three runs.
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Cross-Session Hold-Out: 3 runs (each session as test once)")
    print(f"All {len(ALL_SUBJECTS)} subjects used in each run")

    all_session_results = []
    for i, test_session in enumerate([1, 2, 3]):
        # NEW (pre-session memmap cleanup patch): remove any leftover t_mmap_*.npy files from a
        # previous session before this session's MMapHybridDataset instances are constructed.
        # Under normal operation MMapHybridDataset.cleanup() (called at the end of run_training())
        # already deletes its own memmap files, and every file gets a unique run_tag so a leftover
        # file can never be silently reused/overwritten -- this is an extra safety layer for cases
        # where cleanup() couldn't fully run (e.g. a crashed previous session, or a Windows file
        # lock that prevented deletion), not a replacement for cleanup().
        leftover = glob.glob(os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy"))  # NEW
        if leftover:  # NEW
            print(f"\n[PRE-SESSION CLEANUP] Deleting {len(leftover)} leftover memmap file(s)...")  # NEW
            for f in leftover:  # NEW
                try:  # NEW
                    os.remove(f)  # NEW
                    print(f"  Deleted: {os.path.basename(f)}")  # NEW
                except Exception as e:  # NEW
                    print(f"  WARNING: could not delete {os.path.basename(f)}: {e}")  # NEW
            gc.collect()  # NEW
            if torch.cuda.is_available():  # NEW
                torch.cuda.empty_cache()  # NEW

        result = run_training(weight_decay=0.0, device=device, test_session=test_session)
        all_session_results.append(result)

        # NEW (post-session memmap cleanup patch): double-check that this session's
        # MMapHybridDataset.cleanup() calls (inside run_training()) actually removed their memmap
        # files, and delete any that were missed for whatever reason.
        leftover_after = glob.glob(os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy"))  # NEW
        if leftover_after:  # NEW
            print(f"\n[POST-SESSION CLEANUP] Deleting {len(leftover_after)} missed file(s)...")  # NEW
            for f in leftover_after:  # NEW
                try:  # NEW
                    os.remove(f)  # NEW
                    print(f"  Deleted: {os.path.basename(f)}")  # NEW
                except Exception as e:  # NEW
                    print(f"  WARNING: could not delete {os.path.basename(f)}: {e}")  # NEW
            gc.collect()  # NEW
            if torch.cuda.is_available():  # NEW
                torch.cuda.empty_cache()  # NEW

        # NEW (cooling-break patch): give the hardware (CPU/GPU/disk) a break between sessions --
        # skipped after the last session. cooling_seconds=120 is tuned for a laptop; lower it to
        # 60 on a desktop with better cooling if needed.
        if i < 2:  # NEW: don't cool down after the last session
            cooling_seconds = 120  # NEW
            print(f"\n[COOLING BREAK] Session {test_session} done. Waiting {cooling_seconds}s...")  # NEW
            for remaining in range(cooling_seconds, 0, -10):  # NEW
                print(f"  {remaining}s remaining...", end='\r')  # NEW
                time.sleep(10)  # NEW
            print(f"[COOLING BREAK] Done. Starting next session.\n")  # NEW

    print("\n" + "=" * 70)
    print("CROSS-SESSION HOLD-OUT SUMMARY")
    print("=" * 70)
    print(f"{'test_session':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'kappa':>8}")
    for r in all_session_results:
        print(f"{r['test_session']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['kappa_final']:>8.4f}")

    final_acc   = np.mean([r['test_accuracy_final'] for r in all_session_results])
    final_kappa = np.mean([r['kappa_final']          for r in all_session_results])
    print(f"\nFinal average accuracy (3 sessions): {final_acc:.2f}%")
    print(f"Final average Kappa   (3 sessions): {final_kappa:.4f}")

    summary = {
        'session_results': all_session_results,
        'final_average_accuracy': final_acc,
        'final_average_kappa':   final_kappa,
    }
    summary_path = os.path.join(PROCESSED_DATA_PATH, 'cross_session_A3_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\nFull cross-session summary saved to: {summary_path}")
    print("\nDone.")

Running on Device: cuda
Cross-Session Hold-Out: 3 runs (each session as test once)
All 26 subjects used in each run

######################################################################
# CROSS-SESSION: train on sessions [2, 3]  |  test on session 1
######################################################################
Scanning Train_Set - 26 subjects (sessions [2, 3])...
VP001 verified: 468 windows
VP002 verified: 483 windows
VP003 verified: 531 windows
VP005 verified: 498 windows
VP006 verified: 547 windows
VP007 verified: 457 windows
VP008 verified: 519 windows
VP009 verified: 530 windows
VP010 verified: 510 windows
VP011 verified: 509 windows
VP012 verified: 488 windows
VP013 verified: 472 windows
VP014 verified: 449 windows
VP015 verified: 438 windows
VP016 verified: 523 windows
VP017 verified: 514 windows
VP018 verified: 515 windows
VP019 verified: 505 windows
VP020 verified: 492 windows
VP021 verified: 516 windows
VP022 verified: 502 windows
VP023 verified: 491 windows
VP024 v

In [ ]:
"""

======================================================================
CROSS-SESSION HOLD-OUT SUMMARY
======================================================================
  test_session |  test_acc |  best_val_acc |    kappa
             1 |    62.19% |        53.31% |   0.4330
             2 |    62.84% |        62.83% |   0.4426
             3 |    59.00% |        68.30% |   0.3850

Final average accuracy (3 sessions): 61.34%
Final average Kappa   (3 sessions): 0.4202

"""